# 04. Module Bus & Events — deep dive

> **Read [`03-policy-and-modules.ipynb`](./03-policy-and-modules.ipynb) first.** That notebook covered the *shape* of the bus — `ModuleBus`, `ModuleContext`, the `EventContext`, the policy/bus interplay, and `ToolVetoedError`. **This** notebook is the deep-dive: the *vocabulary* of events that ArcAgent actually emits during a run, the capability lifecycle that ties extensions to the bus, the difference between `HookEntry`, `BackgroundTaskEntry`, and `LifecycleEntry`, three operator-facing extension patterns, and the failure modes you must understand before shipping to a federal deployment.

You will learn:

- The complete event taxonomy `ArcAgent` emits — names, when each fires, and the data payload.
- The capability lifecycle — how `@hook` handlers reach the bus and how a `@capability` class's `setup`/`teardown` run in dependency order.
- When to reach for a `HookEntry` vs a `BackgroundTaskEntry` vs a `LifecycleEntry` — three concepts from `arcagent.capabilities.capability_registry` that look similar but solve different problems.
- Three production patterns: a **metrics** extension, a **structured logging** extension that fans out to a sink, and a **streaming** extension that forwards events to an external transport.
- Failure modes — what happens when a capability's `setup` raises, a handler raises, or a handler times out — and how the bus and loader contain the blast radius.
- How to compose multiple extensions and reason about isolation, ordering, and debugging when something goes wrong.

All cells run with **no API key, no real LLM, no real network**. The bus is in-process, events are synthetic, and any "external" sink is an in-memory list.

## 1. Setup

Standard Arc walkthrough boilerplate — same first cell every notebook in the series uses. After it runs, every package's `src/` (and `tests/`, where present) is on `sys.path`, so `import arcagent`, `import arctrust`, and `import arcrun` resolve from the in-repo source.

> **Reminder.** This is a deep-dive — read [`03-policy-and-modules.ipynb`](./03-policy-and-modules.ipynb) first to understand the bus *surface*. This notebook builds on that foundation.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Pull in the bus primitives, the error types, and a couple of `unittest.mock` helpers we'll use to fake the heavyweight dependencies of `ModuleContext` (`AgentTelemetry`, full `ArcAgentConfig`, etc.) without spinning up a full `ArcAgent`.

In [2]:
import asyncio
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, fields
from typing import Any

import arcagent
from arcagent import ModuleBusError, ToolVetoedError
from arcagent.core.module_bus import (
    EventContext,
    ModuleBus,
    ModuleContext,
)

print("arcagent    :", arcagent.__version__)
print("ModuleBus   :", ModuleBus.__name__)
print("bus surface :", [m for m in dir(ModuleBus) if not m.startswith("_")])

arcagent    : 0.15.0
ModuleBus   : ModuleBus
bus surface : ['emit', 'handler_count', 'handler_count_by_module', 'subscribe']


For a couple of the patterns below we'll also reach for the capability-registry types — `HookEntry`, `BackgroundTaskEntry`, `LifecycleEntry`. These don't sit on the bus directly; they're how the **capability loader** describes the *shape* of an extension before bridging it onto the bus. They live in `arcagent.capabilities.capability_registry` (moved out of `arcagent.core` to keep the nucleus under its LOC budget). Section 4 is dedicated to the distinction.

In [3]:
from arcagent.capabilities.capability_registry import (
    BackgroundTaskEntry,
    HookEntry,
    LifecycleEntry,
)

print("HookEntry            :", HookEntry.__name__)
print("BackgroundTaskEntry  :", BackgroundTaskEntry.__name__)
print("LifecycleEntry       :", LifecycleEntry.__name__)

HookEntry            : HookEntry
BackgroundTaskEntry  : BackgroundTaskEntry
LifecycleEntry       : LifecycleEntry


## 2. The event taxonomy

ArcAgent emits a fixed set of bus events from its core. Bridges (arcrun, arcllm) translate their own event types into bus events as well — the canonical surface is below.

| Event | When it fires | Payload (keys) | Veto allowed? |
|---|---|---|---|
| `agent:ready` | Near the end of `ArcAgent.startup()`, carrying the single deferred-binding callback for extensions that want to drive the agent. | `run_fn`, `deliver_fn`, `skill_registry` | no |
| `agent:init` | Right after `agent:ready`, the last step of `startup()`. | `config` (agent name) | no |
| `agent:pre_respond` | Just before the run loop is launched (one per run). | `task` | no |
| `agent:post_respond` | After the run loop returns, before the agent finalises the session. | `result`, `messages`, `session_id`, `automated` | no |
| `agent:error` | When the run loop raises. | `task`, `error`, `error_type` | no |
| `agent:pre_tool` | Inside `ToolRegistry._create_wrapped_execute`, after policy ALLOW, before tool execution. **This is the veto seam.** | `tool`, `args` | yes — `ctx.veto(reason)` raises `ToolVetoedError` |
| `agent:post_tool` | After successful tool execution. | `tool`, `result`, `duration` | no |
| `agent:pre_plan` | Bridged from arcrun `turn.start`. | arcrun event data | no |
| `agent:post_plan` | Bridged from arcrun `turn.end`. | arcrun event data | no |
| `agent:assemble_prompt` | When `ContextManager.assemble_system_prompt` runs — extensions can inject sections. | `sections`, `workspace`, `query` | no (extensions mutate `sections` in-place via the snapshot) |
| `agent:tools_reloaded` | After `ArcAgent.reload()` re-registers the capability surface. | `{}` | no |
| `agent:shutdown` | First step of `ArcAgent.shutdown()`, before per-component teardown. | `{}` | no |
| `capability:added` | After the capability registry registers a new tool/skill/hook/task/lifecycle. | `kind`, `name`, `version`, `source_path`, `scan_root`, … | no |
| `capability:replaced` | When a registration replaces an existing entry of the same name. | `kind`, `name`, `version`, `previous_version`, … | no |
| `capability:removed` | When `unregister(...)` succeeds. | `kind`, `name`, `version` | no |
| `capability:registration_failed` | The capability loader rejected an entry (validation/conflict). | `path`, `reason`, … | no |
| `capability:registration_warning` | Soft warning from loader (deprecation, fallback). | `path`, `reason`, … | no |
| `capability:setup_failed` | A `LifecycleEntry.instance.setup(ctx)` raised. | `name`, `error`, … | no |
| `llm:call_complete` | Bridged from arcllm `llm_call` traces (and arcrun's mirror). | trace fields incl. `model`, `provider`, `agent_label` | no |
| `llm:config_change` | Bridged from arcllm `config_change`. | trace fields | no |
| `llm:circuit_change` | Bridged from arcllm `circuit_change`. | trace fields | no |

The **only veto seam** is `agent:pre_tool`. Every other emission is observe-only — handlers can read, log, mutate the snapshot dict (it doesn't leak back), forward to a sink, or schedule background work, but they cannot stop the operation.

Let's introspect this in code by registering a *logger module* that captures every event ArcAgent could conceivably emit, then drive synthetic emissions through the same `ModuleBus` and look at the catch list. We won't run a real `ArcAgent` (that requires keys, a workspace, and a full config tree); we drive the bus directly so each event is reproducible and deterministic.

In [4]:
# Synthetic taxonomy probe — register one handler per known event,
# emit a sample of each, see what flows through.

KNOWN_EVENTS: tuple[str, ...] = (
    "agent:ready",
    "agent:init",
    "agent:pre_respond",
    "agent:post_respond",
    "agent:error",
    "agent:pre_tool",
    "agent:post_tool",
    "agent:pre_plan",
    "agent:post_plan",
    "agent:assemble_prompt",
    "agent:tools_reloaded",
    "agent:shutdown",
    "capability:added",
    "capability:replaced",
    "capability:removed",
    "capability:registration_failed",
    "capability:registration_warning",
    "capability:setup_failed",
    "llm:call_complete",
    "llm:config_change",
    "llm:circuit_change",
)

probe_log: list[tuple[str, dict[str, Any]]] = []


async def probe_handler(ctx: EventContext) -> None:
    probe_log.append((ctx.event, dict(ctx.data)))


probe_bus = ModuleBus()
for ev in KNOWN_EVENTS:
    probe_bus.subscribe(ev, probe_handler, module_name="probe")

print("probe handlers registered:", sum(probe_bus.handler_count(e) for e in KNOWN_EVENTS))

probe handlers registered: 21


Now drive synthetic emissions through the bus — same names, same payload shapes the real components would use. The probe captures the catalog.

In [5]:
SAMPLE_PAYLOADS: dict[str, dict[str, Any]] = {
    "agent:ready": {"run_fn": "<callable>", "deliver_fn": "<callable>", "skill_registry": "<registry>"},
    "agent:init": {"config": "demo-agent"},
    "agent:pre_respond": {"task": "summarise the meeting notes"},
    "agent:post_respond": {
        "result": None,
        "messages": [],
        "session_id": "s-1",
        "automated": False,
    },
    "agent:error": {"task": "...", "error": "boom", "error_type": "RuntimeError"},
    "agent:pre_tool": {"tool": "read_file", "args": {"path": "/etc/hosts"}},
    "agent:post_tool": {"tool": "read_file", "result": "127.0.0.1 ...", "duration": 0.012},
    "agent:pre_plan": {"turn": 1},
    "agent:post_plan": {"turn": 1, "actions": 2},
    "agent:assemble_prompt": {
        "sections": {"identity": "..."},
        "workspace": str(Path.cwd() / "_demo_workspace"),
        "query": "summarise the meeting notes",
    },
    "agent:tools_reloaded": {},
    "agent:shutdown": {},
    "capability:added": {"kind": "tool", "name": "search", "version": "1.0.0"},
    "capability:replaced": {
        "kind": "skill",
        "name": "writer",
        "version": "1.1.0",
        "previous_version": "1.0.0",
    },
    "capability:removed": {"kind": "tool", "name": "old_search", "version": "0.9.0"},
    "capability:registration_failed": {"path": "skills/bad/SKILL.md", "reason": "schema invalid"},
    "capability:registration_warning": {
        "path": "skills/old/SKILL.md",
        "reason": "deprecated field",
    },
    "capability:setup_failed": {"name": "memory", "error": "vault unreachable"},
    "llm:call_complete": {
        "event_type": "llm_call",
        "model": "gpt-4o-mini",
        "provider": "openai",
    },
    "llm:config_change": {"event_type": "config_change", "field": "max_tokens"},
    "llm:circuit_change": {"event_type": "circuit_change", "state": "OPEN"},
}

for ev, data in SAMPLE_PAYLOADS.items():
    await probe_bus.emit(ev, data, agent_did="did:arc:demo:operator/abcd")

print("events captured:", len(probe_log))
print("first 5 captured events:")
for name, data in probe_log[:5]:
    print(f"  {name:>34}  {data}")

events captured: 21
first 5 captured events:
                         agent:ready  {'run_fn': '<callable>', 'deliver_fn': '<callable>', 'skill_registry': '<registry>'}
                          agent:init  {'config': 'demo-agent'}
                   agent:pre_respond  {'task': 'summarise the meeting notes'}
                  agent:post_respond  {'result': None, 'messages': [], 'session_id': 's-1', 'automated': False}
                         agent:error  {'task': '...', 'error': 'boom', 'error_type': 'RuntimeError'}


The catch list confirms that **the bus is content-agnostic** — it dispatches whatever name + dict you emit. The "taxonomy" is a contract enforced by the *callers* in `arcagent.core.*`, not by the bus itself. This matters for two reasons:

1. **Modules can subscribe to events that haven't fired yet.** No registration error if the name is mistyped — silence. Always assert at least one event lands during integration tests.
2. **Custom modules can introduce their own namespaces** (e.g. `myorg:metrics_flush`) and the bus will dispatch them. Use a colon-prefixed namespace — never collide with `agent:*`, `capability:*`, or `llm:*`.

In [6]:
# Distribution by namespace
buckets = Counter(name.split(":", 1)[0] for name, _ in probe_log)
for ns, count in sorted(buckets.items()):
    print(f"  {ns:>12} {count:>3} events")

         agent  12 events
    capability   6 events
           llm   3 events


## 3. The capability lifecycle — subscribe, dispatch, setup/teardown

There are three independent axes here — don't conflate them:

**1. How handlers get onto the bus.** An extension's event handlers reach
the bus through `bus.subscribe(event, handler, priority=..., module_name=...)`.
In production those handlers are `@hook(event=...)`-decorated functions the
`CapabilityLoader` discovers on disk; the loader's hook bridge calls
`subscribe(...)` for each. There is no `register_module`/`startup`/`shutdown`
on the bus — it is a pure subscribe/dispatch fabric.

**2. Resource lifecycle (`@capability` classes).** A capability that owns a
resource — a connection pool, a file handle, a telemetry exporter — is a class
decorated with `@capability`, exposing `async setup(ctx)` and `async teardown()`.
The `CapabilityLoader` drives them:

```
loader.start_lifecycles()  ──▶  for entry in topological_order(depends_on):
                                     await entry.instance.setup(ctx)
                                 (on failure: emit capability:setup_failed,
                                  roll back set-up siblings in reverse, re-raise)

loader.shutdown()          ──▶  for entry in reversed(topological_order):
                                     await entry.instance.teardown()
```

**3. Handler dispatch (a third, orthogonal axis).** When `bus.emit(event, data)`
fires, handlers run grouped by `priority` — `10` before `50` before `100`
before `200`; within a group they run concurrently via `asyncio.gather`.

Two guarantees worth memorising:

1. **Setup runs in dependency-topological order** (`@capability(depends_on=...)`),
   and **teardown runs in reverse** — the same acquire/release discipline as
   Python's `contextlib.ExitStack`.
2. **Handlers within the same priority level run concurrently**, but priority
   groups are processed sequentially low-first. A handler at priority `10`
   always finishes before any handler at priority `50` starts, even if it's the
   slowest one in its group.

Here's a concrete demonstration of the resource lifecycle. Three `@capability` classes — `alpha`, `beta`, `gamma` — each logs its own `setup`/`teardown`. We drive them exactly the way `CapabilityLoader.start_lifecycles()` / `shutdown()` do: `setup` in order, `teardown` in reverse.

In [7]:
from arcagent.tools import capability


def make_trace_capability(cap_name: str, log: list[str]):
    """A @capability class logs its own lifecycle into a shared list.

    ``@capability`` stamps lifecycle metadata; the class exposes the
    ``setup(ctx)`` / ``teardown()`` contract the loader drives.
    """

    @capability(name=cap_name)
    class TraceCapability:
        async def setup(self, ctx: Any) -> None:
            log.append(f"setup:{cap_name}")

        async def teardown(self) -> None:
            log.append(f"teardown:{cap_name}")

    return TraceCapability()


lifecycle_log: list[str] = []
caps = [make_trace_capability(n, lifecycle_log) for n in ("alpha", "beta", "gamma")]

# Mirror CapabilityLoader.start_lifecycles(): setup in topological order.
# With no depends_on, discovery order is preserved. The loader passes ctx=None
# in this path; a real ModuleContext arrives when the agent wires it.
for cap in caps:
    await cap.setup(None)

# Mirror CapabilityLoader.shutdown(): teardown in reverse.
for cap in reversed(caps):
    await cap.teardown()

print("lifecycle trace:")
for line in lifecycle_log:
    print(f"  {line}")

lifecycle trace:
  setup:alpha
  setup:beta
  setup:gamma
  teardown:gamma
  teardown:beta
  teardown:alpha


Two things are worth highlighting in that trace:

- Setups run **alpha → beta → gamma** (topological order; with no `depends_on`
  the loader preserves discovery order).
- Teardowns run **gamma → beta → alpha** (reverse).

This isn't decoration — it's required for clean teardown. If `gamma`'s `setup`
acquired a handle that depends on `alpha`'s resource still being open, then
`alpha`'s `teardown` must run *after* `gamma`'s — which is exactly what reversed
iteration gives you. `start_lifecycles()` adds one more guarantee: if a `setup`
raises, it emits `capability:setup_failed`, tears down the already-set-up
siblings in reverse, and re-raises (see §8.1).

**Handler dispatch ordering** is a separate axis from the capability lifecycle. Inside a single emit, handlers are sorted by `priority` and dispatched in groups. Demonstration:

In [8]:
dispatch_order: list[str] = []


async def make_h(label: str) -> Any:
    async def handler(ctx: EventContext) -> None:
        dispatch_order.append(label)

    return handler


prio_bus = ModuleBus()
prio_bus.subscribe("ev", await make_h("p100-default"), priority=100)
prio_bus.subscribe("ev", await make_h("p050-security"), priority=50)
prio_bus.subscribe("ev", await make_h("p010-policy"), priority=10)
prio_bus.subscribe("ev", await make_h("p200-logging"), priority=200)
prio_bus.subscribe("ev", await make_h("p010-policy-2"), priority=10)

await prio_bus.emit("ev", {})

print("dispatch order:")
for label in dispatch_order:
    print(f"  {label}")

dispatch order:
  p010-policy
  p010-policy-2
  p050-security
  p100-default
  p200-logging


Notice the two `p010-policy*` handlers can appear in either order relative to each other (same priority → concurrent), but they always finish before any `p050` handler starts.

**Convention** (used by Arc's built-in modules):

| Priority | Use |
|---|---|
| `10` | Policy / authorization (must run first; can veto on `agent:pre_tool`) |
| `50` | Security / sanitisation (input redaction, output filtering) |
| `100` | **Default** — observation, telemetry, capability bridging |
| `200` | Logging, audit, "slow" sinks (don't block faster handlers) |

## 4. HookEntry vs BackgroundTaskEntry vs LifecycleEntry

These three look superficially similar — they all live in `arcagent.capabilities.capability_registry` and they're all "things an extension can register" — but they answer **different questions**:

| Type | Question it answers | Lifecycle | Cardinality |
|---|---|---|---|
| **`HookEntry`** | "When **event X** fires, run **this function**." | One-shot — invoked per event emission. | Many per event (fan-out, ordered by `priority`). |
| **`BackgroundTaskEntry`** | "Run this coroutine **continuously** for the agent's lifetime." | Long-running — `asyncio.Task` started on registration; cancelled-and-awaited on replace or shutdown. | One per name. |
| **`LifecycleEntry`** | "When the **agent** starts/stops, call **this object's** `setup`/`teardown`." | Two-shot — `setup(ctx)` once, `teardown()` once. | One per name. |

In other words:

- **Hooks** are *event-driven*. They sleep until an event fires.
- **Background tasks** are *time-driven*. They spin in their own task, doing whatever work needs to happen on a schedule, with their own backpressure.
- **Lifecycle entries** are *state-bound*. They model objects that need explicit acquire/release semantics — file handles, connection pools, telemetry exporters.

In `03-policy-and-modules.ipynb` we wrote an extension that subscribed handlers to the bus directly. The capability registry types are the *declarative* layer the **capability loader** uses when scanning capability folders — `@hook(event="X")`, `@background_task(...)`, `@capability(...)` decorators produce these entries, and the loader registers them and bridges the hooks onto the bus on the agent's behalf.

Let's look at the data shape of each. Field names tell you what each entry is responsible for tracking — start by inspecting the dataclass fields.

In [9]:
def show_fields(cls: type) -> None:
    print(f"{cls.__name__}:")
    for f in fields(cls):
        print(f"    {f.name:>14}: {f.type}")


show_fields(HookEntry)
print()
show_fields(BackgroundTaskEntry)
print()
show_fields(LifecycleEntry)

HookEntry:
              meta: HookMetadata
           handler: Callable[..., Awaitable[None]]
       source_path: Path
         scan_root: str

BackgroundTaskEntry:
              meta: BackgroundTaskMetadata
                fn: Callable[..., Coroutine[Any, Any, None]]
       source_path: Path
         scan_root: str
              task: asyncio.Task[None] | None

LifecycleEntry:
              meta: CapabilityClassMetadata
          instance: object
       source_path: Path
         scan_root: str
        setup_done: bool


Decision matrix — when to reach for each:

```
          ┌────────────────────────────────────────────────┐
          │ Is the work triggered by something the agent   │
          │ already does (a tool call, a turn boundary,    │
          │ prompt assembly)?                              │
          └─────────────────┬──────────────────────────────┘
                            │ yes
                            ▼
                    ┌──────────────┐
                    │  HookEntry   │  @hook(event="agent:post_tool")
                    └──────────────┘
                            │ no
                            ▼
          ┌────────────────────────────────────────────────┐
          │ Is the work continuous/periodic with its own   │
          │ schedule, independent of agent activity?       │
          └─────────────────┬──────────────────────────────┘
                            │ yes
                            ▼
                    ┌─────────────────────────┐
                    │  BackgroundTaskEntry    │  @background_task(interval=30)
                    └─────────────────────────┘
                            │ no
                            ▼
          ┌────────────────────────────────────────────────┐
          │ Does the work need explicit setup/teardown to  │
          │ acquire and release a resource?                │
          └─────────────────┬──────────────────────────────┘
                            │ yes
                            ▼
                    ┌─────────────────────┐
                    │  LifecycleEntry     │  @capability  (class with setup/teardown)
                    └─────────────────────┘
```

A `@capability` class (with `setup`/`teardown`) *is* a `LifecycleEntry`; it can
also carry `@tool`- or `@hook`-decorated methods that the loader binds to the
same instance. Most extensions skip the class entirely and declare `@hook` /
`@background_task` primitives at module level — the loader bridges them to the
bus for you.

## 5. Pattern: a metrics module

**Goal.** Operator wants per-tool counts and timings, plus error totals. The numbers must be cheap to read at any moment (for a `/metrics` endpoint, a status page, or a Prometheus scrape).

**Design.** Subscribe to the three events that carry the relevant signal:

- `agent:pre_tool` → increment a "tools called" counter, stamp a start time on the trace_id.
- `agent:post_tool` → take elapsed (already in payload as `duration`), add to the histogram, increment "tools succeeded".
- `agent:error` → increment "errors total", bucket by `error_type`.

Expose a single `snapshot()` method that returns a frozen dict — operators read it however they like.

In [10]:
class MetricsModule:
    """Per-tool counts + duration sums + error totals.

    Reads:  agent:pre_tool, agent:post_tool, agent:error
    Writes: in-memory counters; expose via snapshot().
    Install by subscribing handlers to the bus at priority 200 (observation
    belongs at the back of the queue) — exactly what the @hook bridge does.
    """

    NAME = "metrics"

    def __init__(self) -> None:
        self.calls: Counter[str] = Counter()
        self.successes: Counter[str] = Counter()
        self.duration_sum: dict[str, float] = defaultdict(float)
        self.errors: Counter[str] = Counter()

    def install(self, bus: ModuleBus) -> None:
        bus.subscribe("agent:pre_tool", self._on_pre_tool, priority=200, module_name=self.NAME)
        bus.subscribe("agent:post_tool", self._on_post_tool, priority=200, module_name=self.NAME)
        bus.subscribe("agent:error", self._on_error, priority=200, module_name=self.NAME)

    async def _on_pre_tool(self, ctx: EventContext) -> None:
        self.calls[ctx.data.get("tool", "?")] += 1

    async def _on_post_tool(self, ctx: EventContext) -> None:
        tool = ctx.data.get("tool", "?")
        self.successes[tool] += 1
        self.duration_sum[tool] += float(ctx.data.get("duration", 0.0))

    async def _on_error(self, ctx: EventContext) -> None:
        self.errors[ctx.data.get("error_type", "Unknown")] += 1

    def snapshot(self) -> dict[str, Any]:
        return {
            "calls": dict(self.calls),
            "successes": dict(self.successes),
            "duration_sum_seconds": dict(self.duration_sum),
            "errors": dict(self.errors),
        }

Wire it onto a bus and drive a synthetic workload — three calls to `read_file`, two to `bash`, one transient error.

In [11]:
metrics_bus = ModuleBus()
metrics = MetricsModule()
metrics.install(metrics_bus)


async def fake_dispatch(bus: ModuleBus, tool: str, duration: float) -> None:
    await bus.emit("agent:pre_tool", {"tool": tool, "args": {}})
    await bus.emit("agent:post_tool", {"tool": tool, "result": "ok", "duration": duration})


for d in (0.012, 0.020, 0.011):
    await fake_dispatch(metrics_bus, "read_file", d)
for d in (0.250, 0.310):
    await fake_dispatch(metrics_bus, "bash", d)

# Synthetic error
await metrics_bus.emit(
    "agent:error",
    {"task": "...", "error": "kaboom", "error_type": "TimeoutError"},
)

snap = metrics.snapshot()
print("calls          :", snap["calls"])
print("successes      :", snap["successes"])
print("duration_sum_s :", {k: round(v, 4) for k, v in snap["duration_sum_seconds"].items()})
print("errors         :", snap["errors"])

calls          : {'read_file': 3, 'bash': 2}
successes      : {'read_file': 3, 'bash': 2}
duration_sum_s : {'read_file': 0.043, 'bash': 0.56}
errors         : {'TimeoutError': 1}


**Operational note.** This extension already wires its handlers at
**priority 200** so they sit behind policy and security handlers —
observation belongs at the back of the priority queue, never in front of a
handler that might veto or sanitise.

## 6. Pattern: a structured logging module

**Goal.** Every interesting event turns into a structured log line going to a sink. The sink could be a file, a `JsonlSink`, a `SignedChainSink` from `arctrust.audit`, or stdout in dev. The module shouldn't care — it should fan out to whatever the operator hands it.

**Cross-reference.** This is the *same shape* as `arctrust.audit.emit(AuditEvent, sink)` in `arctrust/04-audit-sinks.ipynb`. The bus-level logger is for *operational* visibility (what the agent did and when); the audit sink is for *forensic* visibility (tamper-evident, schema-frozen). Use both. They answer different questions.

In [12]:
@dataclass(frozen=True)
class LogRecord:
    timestamp: float
    event: str
    agent_did: str
    payload: dict[str, Any]


class InMemorySink:
    """Stand-in for JsonlSink / SignedChainSink. Tests use this; prod swaps for real."""

    def __init__(self) -> None:
        self.records: list[LogRecord] = []

    def write(self, record: LogRecord) -> None:
        self.records.append(record)


class StructuredLoggerModule:
    """Fan every interesting bus event out to a sink as a structured record."""

    NAME = "structured_logger"

    # Events the operator actually cares about — keep this list small.
    SUBSCRIBED_EVENTS = (
        "agent:pre_tool",
        "agent:post_tool",
        "agent:error",
        "capability:added",
        "capability:replaced",
        "capability:removed",
    )

    def __init__(self, sink: InMemorySink) -> None:
        self._sink = sink

    def install(self, bus: ModuleBus) -> None:
        for ev in self.SUBSCRIBED_EVENTS:
            bus.subscribe(ev, self._log, priority=200, module_name=self.NAME)

    async def _log(self, ctx: EventContext) -> None:
        self._sink.write(
            LogRecord(
                timestamp=time.monotonic(),
                event=ctx.event,
                agent_did=ctx.agent_did,
                payload=dict(ctx.data),
            )
        )

Stand it up, drive a representative workload, and inspect the sink.

In [13]:
log_bus = ModuleBus()
sink = InMemorySink()
logger_mod = StructuredLoggerModule(sink)
logger_mod.install(log_bus)

# Drive synthetic events the operator would care about
DID = "did:arc:demo:operator/abcd1234"
await log_bus.emit("agent:pre_tool", {"tool": "read_file", "args": {"path": "x"}}, agent_did=DID)
await log_bus.emit(
    "agent:post_tool", {"tool": "read_file", "result": "...", "duration": 0.014}, agent_did=DID
)
await log_bus.emit(
    "capability:added",
    {"kind": "tool", "name": "search", "version": "1.0.0"},
    agent_did=DID,
)
await log_bus.emit(
    "agent:error",
    {"task": "...", "error": "kaboom", "error_type": "TimeoutError"},
    agent_did=DID,
)

print(f"sink records: {len(sink.records)}")
for r in sink.records:
    print(f"  {r.event:>26}  did={r.agent_did[-8:]}  payload_keys={sorted(r.payload)}")

sink records: 4
              agent:pre_tool  did=abcd1234  payload_keys=['args', 'tool']
             agent:post_tool  did=abcd1234  payload_keys=['duration', 'result', 'tool']
            capability:added  did=abcd1234  payload_keys=['kind', 'name', 'version']
                 agent:error  did=abcd1234  payload_keys=['error', 'error_type', 'task']


**Why a separate module instead of "just call telemetry directly"?** Two reasons:

1. **Separation of concerns.** `AgentTelemetry` is a *core* component that emits a fixed set of audit events. The structured logger is *operator-facing*: which events to capture, the destination sink, the redaction policy — all of that is configurable per deployment without touching core.
2. **Hot-swappable sinks.** Federal swaps `InMemorySink` for `arctrust.audit.SignedChainSink`. Personal swaps it for stdout. Same module body, different config. The bus is the integration seam.

## 7. Pattern: a streaming module (events to an external transport)

**Goal.** Forward live events to an external transport — a websocket, a NATS subject, a queue worker. Anything that needs to see what the agent is doing **in real time**, not after the run finishes.

**Design constraints.**

- The bus dispatches handlers under a 30-second default timeout. A blocking transport call would burn that budget. **Push to a local queue, drain in a background task.**
- The transport may be down. Drop or buffer? Operator policy. We model both.
- Backpressure must not stall the bus. If the queue is full, drop and increment a counter — never await a long send while a tool dispatch is waiting on the post_tool emit.

In [14]:
class StreamingModule:
    """Forward events to an external transport via a bounded asyncio.Queue.

    Hot path (in-handler) is non-blocking: enqueue or drop.
    Drain runs as a background task, started in install(). This is the
    in-process shape of a @background_task capability.
    """

    NAME = "streaming"
    SUBSCRIBED_EVENTS = ("agent:pre_tool", "agent:post_tool", "agent:error")

    def __init__(
        self, transport: list[dict[str, Any]], *, capacity: int = 16, send_latency: float = 0.0
    ) -> None:
        # Mock transport: a list. In production this is a websocket, NATS, etc.
        self._transport = transport
        self._queue: asyncio.Queue[dict[str, Any]] = asyncio.Queue(maxsize=capacity)
        self._dropped = 0
        self._send_latency = send_latency
        self._drain_task: asyncio.Task[None] | None = None

    @property
    def dropped(self) -> int:
        return self._dropped

    def install(self, bus: ModuleBus) -> None:
        for ev in self.SUBSCRIBED_EVENTS:
            bus.subscribe(ev, self._enqueue, module_name=self.NAME)
        self._drain_task = asyncio.create_task(self._drain_loop())

    async def aclose(self) -> None:
        if self._drain_task is not None:
            self._drain_task.cancel()
            try:
                await self._drain_task
            except asyncio.CancelledError:
                pass

    async def _enqueue(self, ctx: EventContext) -> None:
        envelope = {"event": ctx.event, "data": dict(ctx.data), "agent_did": ctx.agent_did}
        try:
            self._queue.put_nowait(envelope)
        except asyncio.QueueFull:
            self._dropped += 1

    async def _drain_loop(self) -> None:
        while True:
            envelope = await self._queue.get()
            # In production this would be `await ws.send_json(envelope)` etc.
            # ``send_latency`` simulates a transport slower than the emit rate —
            # that's what makes the bounded queue overflow and drop. The drain
            # task can take as long as it needs; it's not on the hot path, so a
            # slow transport doesn't slow tool dispatch.
            if self._send_latency:
                await asyncio.sleep(self._send_latency)
            self._transport.append(envelope)
            self._queue.task_done()

Drive a workload — emit faster than the drain task can keep up to demonstrate backpressure handling.

In [15]:
stream_bus = ModuleBus()
transport: list[dict[str, Any]] = []
# capacity=4 with a slow (5 ms/event) transport: the burst overruns the queue.
streaming = StreamingModule(transport, capacity=4, send_latency=0.005)
streaming.install(stream_bus)

# Burst more events than capacity to exercise the drop path
for i in range(20):
    await stream_bus.emit(
        "agent:pre_tool", {"tool": "echo", "args": {"i": i}}, agent_did="did:demo"
    )

# Let the drain task catch up (drains whatever survived the overrun)
await asyncio.sleep(0.1)

print("events emitted     : 20")
print("transport received :", len(transport))
print("queue dropped      :", streaming.dropped)
print("received + dropped :", len(transport) + streaming.dropped)
print("first 3 forwarded  :")
for env in transport[:3]:
    print(f"  {env['event']}  {env['data']}")

await streaming.aclose()

events emitted     : 20
transport received : 5
queue dropped      : 15
received + dropped : 20
first 3 forwarded  :
  agent:pre_tool  {'tool': 'echo', 'args': {'i': 0}}
  agent:pre_tool  {'tool': 'echo', 'args': {'i': 1}}
  agent:pre_tool  {'tool': 'echo', 'args': {'i': 2}}


**Production checklist** for a streaming module like this:

- Use a **bounded** queue. Unbounded queues mask leaks and blow up memory.
- Decide drop policy explicitly. Drop newest? Drop oldest? Coalesce? Operator-tunable via config.
- Emit a `metrics:queue_depth` event (your own namespace!) at intervals so the metrics module can see backpressure.
- On `agent:shutdown`, *drain* the queue before cancelling the task — otherwise tail events are lost.

## 8. Failure modes

### 8.1 A capability's `setup` raises

`CapabilityLoader.start_lifecycles()` runs each `@capability`'s `setup(ctx)`
in topological order. If one raises, the loader:

1. emits `capability:setup_failed` (so the UI reporter / audit sink sees it),
2. tears down the already-set-up siblings in **reverse** order, and
3. **re-raises** — a failed capability setup is fail-loud, because a half-wired
   resource graph is worse than no agent.

This is different from *handler* failures (§8.2 / §8.3), which are swallowed. A
capability is infrastructure; a handler is an observer. Below we drive a mini
mirror of `start_lifecycles()` to show the containment.

In [16]:
class BrokenCapability:
    """setup() raises — simulates a vault/identity failure."""

    NAME = "broken"

    async def setup(self, ctx: Any) -> None:
        raise RuntimeError("vault unreachable; cannot acquire signing key")

    async def teardown(self) -> None:
        return None


class HealthyCapability:
    NAME = "healthy"

    def __init__(self) -> None:
        self.torn_down = False

    async def setup(self, ctx: Any) -> None:
        return None

    async def teardown(self) -> None:
        self.torn_down = True


async def start_lifecycles(bus: ModuleBus, caps: list[Any]) -> None:
    """Mini mirror of CapabilityLoader.start_lifecycles()."""
    set_up: list[Any] = []
    for cap in caps:
        try:
            await cap.setup(None)
            set_up.append(cap)
        except Exception as exc:
            await bus.emit("capability:setup_failed", {"name": cap.NAME, "error": str(exc)})
            for done in reversed(set_up):  # roll back already-set-up siblings
                await done.teardown()
            raise


setup_failed_events: list[dict[str, Any]] = []


async def _record_setup_failed(ctx: EventContext) -> None:
    setup_failed_events.append(dict(ctx.data))


fail_bus = ModuleBus()
fail_bus.subscribe("capability:setup_failed", _record_setup_failed)

healthy = HealthyCapability()
broken = BrokenCapability()

# healthy sets up, then broken raises → rollback tears healthy down, re-raise.
try:
    await start_lifecycles(fail_bus, [healthy, broken])
    print("UNEXPECTED — start_lifecycles did not raise")
except RuntimeError as exc:
    print("capability setup failed and re-raised:", exc)

print("capability:setup_failed events :", setup_failed_events)
print("healthy rolled back (torn_down):", healthy.torn_down)

capability setup failed and re-raised: vault unreachable; cannot acquire signing key
capability:setup_failed events : [{'name': 'broken', 'error': 'vault unreachable; cannot acquire signing key'}]
healthy rolled back (torn_down): True


Note the containment: `capability:setup_failed` fired so observers know, and
the already-set-up `healthy` capability was **torn down in reverse** — no
resource is left half-acquired. Unlike a handler failure, the exception
propagates, so the agent refuses to finish starting with a broken
infrastructure capability.

If you want to *convert* that into a structured, catchable error at the agent
boundary — for example to attach operational context and route it to your
alerting — wrap the call and raise `ModuleBusError`:

In [17]:
# Pattern: convert a fail-loud capability setup failure into a structured
# ModuleBusError at the agent boundary. We catch it below so the notebook still
# runs end-to-end — in your own code you'd let it propagate to the caller.
critical_bus = ModuleBus()
critical_bus.subscribe("capability:setup_failed", _record_setup_failed)

try:
    try:
        await start_lifecycles(critical_bus, [BrokenCapability()])
    except RuntimeError as exc:
        raise ModuleBusError(
            code="CRITICAL_CAPABILITY_SETUP_FAILED",
            message=f"Capability 'broken' failed; refusing to continue: {exc}",
            details={"capability": "broken"},
        ) from exc
except ModuleBusError as exc:
    print("fail-loud surfaced:", exc)
    print("code             :", exc.code)
    print("details          :", exc.details)

fail-loud surfaced: [CRITICAL_CAPABILITY_SETUP_FAILED] module_bus: Capability 'broken' failed; refusing to continue: vault unreachable; cannot acquire signing key
code             : CRITICAL_CAPABILITY_SETUP_FAILED
details          : {'capability': 'broken'}


### 8.2 Handler raises during emit

Same containment story, different scope. The bus catches the exception inside `_run_handler`, logs it, and continues. Other handlers in the same priority group still complete, the next priority group still runs, the emit caller still gets a non-vetoed `EventContext` back.

**The implication:** handler exceptions are **invisible** to upstream code. If your metrics module crashes on `agent:post_tool`, you won't see broken metrics in the agent's call site — you'll see *missing* numbers. Always log exceptions at the module level too if you care about observability.

In [18]:
raise_log: list[str] = []


async def good_handler(ctx: EventContext) -> None:
    raise_log.append("good ran")


async def bad_handler(ctx: EventContext) -> None:
    raise_log.append("bad ran (will raise next)")
    raise RuntimeError("simulated handler crash")


async def also_good_handler(ctx: EventContext) -> None:
    raise_log.append("also_good ran")


emit_bus = ModuleBus()
emit_bus.subscribe("ev", good_handler)
emit_bus.subscribe("ev", bad_handler)
emit_bus.subscribe("ev", also_good_handler)

result = await emit_bus.emit("ev", {})
print("module failed during emit: bad_handler raised, others kept running")
print("captured trace :", raise_log)
print("emit returned  :", type(result).__name__, "vetoed=", result.is_vetoed)

Handler bad_handler for event ev raised an exception
Traceback (most recent call last):
  File "/Users/joshschultz/Projects/arc/.claude/worktrees/agent-af67a1128a112c184/packages/arcagent/src/arcagent/core/module_bus.py", line 167, in _run_handler
    await asyncio.wait_for(
  File "/Users/joshschultz/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
  File "/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/ipykernel_22956/497232950.py", line 10, in bad_handler
    raise RuntimeError("simulated handler crash")
RuntimeError: simulated handler crash


module failed during emit: bad_handler raised, others kept running
captured trace : ['good ran', 'bad ran (will raise next)', 'also_good ran']
emit returned  : EventContext vetoed= False


### 8.3 Handler times out

Handlers run under a per-registration timeout (`timeout_seconds`, default 30s). A timeout is logged at `WARNING` and the handler is dropped — same as a crash. The next group still runs.

This is **why metrics/logging modules belong at priority 200**: a slow logging sink should never starve a faster security check at priority 50.

In [19]:
timeout_log: list[str] = []


async def slow(ctx: EventContext) -> None:
    timeout_log.append("slow started")
    await asyncio.sleep(1.0)  # > 100ms timeout below
    timeout_log.append("slow finished — should NOT appear")


async def fast(ctx: EventContext) -> None:
    timeout_log.append("fast ran")


timeout_bus = ModuleBus()
timeout_bus.subscribe("ev", slow, timeout_seconds=0.05)
timeout_bus.subscribe("ev", fast, priority=200)

t0 = time.monotonic()
await timeout_bus.emit("ev", {})
elapsed_ms = (time.monotonic() - t0) * 1000
print(f"emit elapsed: {elapsed_ms:.0f} ms (slow handler hit 50ms timeout)")
print("trace:", timeout_log)

Handler slow for event ev timed out after 0.1s


emit elapsed: 52 ms (slow handler hit 50ms timeout)
trace: ['slow started', 'fast ran']


### 8.4 Mapping exception types to surfaces

Quick reference card:

| Trigger | Surface | Subclass of | Behaviour |
|---|---|---|---|
| Policy DENY | `PolicyDenied` | `ArcAgentError` | Bus never sees the call. |
| `ctx.veto(reason)` on `agent:pre_tool` | `ToolVetoedError` | `ToolError` → `ArcAgentError` | Bus dispatched all handlers; first veto wins. |
| Caller-raised lifecycle assertion | `ModuleBusError` | `ArcAgentError` | Caller-emitted; the bus never raises this itself. |
| Handler crashes / times out | (logged, swallowed) | n/a | All other handlers in the same priority group still run. |
| `@capability` `setup` raises | `capability:setup_failed` event, then the original exception re-raised | n/a | Loader tears down set-up siblings in reverse; a caller may wrap in `ModuleBusError`. |

`ModuleBusError` is **not** raised by the bus on handler failure — that's the whole point of the isolation. It exists so *callers* (the agent, your operational tooling) can wrap bus/capability interactions in structured exceptions when they need fail-loud semantics.

In [20]:
err = ModuleBusError(
    code="CRITICAL_CAPABILITY_SETUP_FAILED",
    message="Capability 'vault' failed to set up: vault unreachable",
    details={"capability": "vault", "phase": "setup"},
)
print("str(err)  :", err)
print("code      :", err.code)
print("component :", err.component)
print("details   :", err.details)

str(err)  : [CRITICAL_CAPABILITY_SETUP_FAILED] module_bus: Capability 'vault' failed to set up: vault unreachable
code      : CRITICAL_CAPABILITY_SETUP_FAILED
component : module_bus
details   : {'capability': 'vault', 'phase': 'setup'}


## 9. Composition — multiple modules in concert

Real deployments stack three or more modules. The interesting questions become:

- **What's the dispatch order?** (priorities, then registration order within a priority)
- **What does each module see?** (they all see the same `EventContext` snapshot — but mutations to `ctx.data` in one handler do not leak to the next emit)
- **How do you debug "wrong number" or "missing event"?** (`bus.handler_count(event)` and per-module module_name tagging)

Let's run the metrics module, the logger, and the streaming module together against a synthetic workload that mimics one tool-heavy agent run.

In [21]:
# Compose all three production extensions.
combo_bus = ModuleBus()
combo_metrics = MetricsModule()
combo_sink = InMemorySink()
combo_logger = StructuredLoggerModule(combo_sink)
combo_transport: list[dict[str, Any]] = []
combo_streaming = StreamingModule(combo_transport, capacity=32)

# Install order is cosmetic — each subscribes independently, and priorities
# (not install order) decide dispatch order.
combo_metrics.install(combo_bus)
combo_logger.install(combo_bus)
combo_streaming.install(combo_bus)

print("extensions :", [MetricsModule.NAME, StructuredLoggerModule.NAME, StreamingModule.NAME])
for ev in ("agent:pre_tool", "agent:post_tool", "agent:error", "capability:added"):
    print(f"  {ev:>24} handlers: {combo_bus.handler_count(ev)}")

extensions : ['metrics', 'structured_logger', 'streaming']
            agent:pre_tool handlers: 3
           agent:post_tool handlers: 3
               agent:error handlers: 3
          capability:added handlers: 1


Drive a small synthetic run — five tool calls, one error, one capability registration. Each event fans out to all three extensions in parallel (within priority), then we read each one's view.

In [22]:
DID = "did:arc:demo:operator/abcd1234"

for i in range(3):
    await combo_bus.emit("agent:pre_tool", {"tool": "read_file", "args": {"i": i}}, agent_did=DID)
    await combo_bus.emit(
        "agent:post_tool", {"tool": "read_file", "result": "ok", "duration": 0.01}, agent_did=DID
    )

for i in range(2):
    await combo_bus.emit("agent:pre_tool", {"tool": "bash", "args": {"i": i}}, agent_did=DID)
    await combo_bus.emit(
        "agent:post_tool", {"tool": "bash", "result": "ok", "duration": 0.25}, agent_did=DID
    )

await combo_bus.emit(
    "agent:error", {"task": "x", "error": "boom", "error_type": "RuntimeError"}, agent_did=DID
)
await combo_bus.emit(
    "capability:added",
    {"kind": "tool", "name": "search", "version": "1.0.0"},
    agent_did=DID,
)

# Let the streaming drain task catch up
await asyncio.sleep(0.05)

print("=== metrics ===")
print(combo_metrics.snapshot())
print()
print("=== structured log records ===")
print(f"count: {len(combo_sink.records)}")
for r in combo_sink.records[-3:]:
    print(f"  ... {r.event:>26}  payload_keys={sorted(r.payload)}")
print()
print("=== streaming transport ===")
print(f"forwarded: {len(combo_transport)}, dropped: {combo_streaming.dropped}")

await combo_streaming.aclose()

=== metrics ===
{'calls': {'read_file': 3, 'bash': 2}, 'successes': {'read_file': 3, 'bash': 2}, 'duration_sum_seconds': {'read_file': 0.03, 'bash': 0.5}, 'errors': {'RuntimeError': 1}}

=== structured log records ===
count: 12
  ...            agent:post_tool  payload_keys=['duration', 'result', 'tool']
  ...                agent:error  payload_keys=['error', 'error_type', 'task']
  ...           capability:added  payload_keys=['kind', 'name', 'version']

=== streaming transport ===
forwarded: 11, dropped: 0


**Debugging composition.** When something looks off, walk this checklist:

1. `bus.handler_count(event)` — is the count what you expect for that event?
2. Are handlers tagged with `module_name=...` so the bus's debug log (and `handler_count_by_module`) identifies *which* extension's handler crashed/timed-out?
3. Did a capability's `setup` fail? Those re-raise (§8.1) and emit `capability:setup_failed` — check your logs/sink for it, and remember a failed setup means the agent never finished starting.
4. Are priorities collision-free? Two priority-100 handlers run concurrently; if they race on shared state, you have a bug *in the extensions*, not the bus.
5. Is your handler async? Sync handlers can't be awaited and the bus's `_run_handler` will raise `TypeError` on first dispatch (logged, swallowed).

## 10. Summary

**Event taxonomy.** `ArcAgent` emits a fixed catalog of events through the `ModuleBus`: `agent:ready`, `agent:init`, `agent:pre_respond` / `post_respond`, `agent:error`, `agent:pre_tool` / `post_tool`, `agent:pre_plan` / `post_plan`, `agent:assemble_prompt`, `agent:tools_reloaded`, `agent:shutdown`, `capability:added` / `replaced` / `removed` / `registration_failed` / `registration_warning` / `setup_failed`, `llm:call_complete` / `config_change` / `circuit_change`. **Only `agent:pre_tool` accepts a veto.** Everything else is observe-only.

**Two axes, don't conflate them.** The **bus** is a pure subscribe/dispatch fabric — `subscribe` + `emit`, nothing else. The **capability lifecycle** (`@capability` classes with `setup`/`teardown`) is driven by the `CapabilityLoader`: `start_lifecycles()` runs `setup` in dependency-topological order; `shutdown()` runs `teardown` in reverse. Within a single `emit`, handlers dispatch by `priority` — `10` before `50` before `100` before `200`; same-priority handlers run concurrently via `asyncio.gather`.

**Three "things you can register".**

| Concept | Trigger | Cardinality |
|---|---|---|
| `HookEntry` | Bus event fires | Many per event |
| `BackgroundTaskEntry` | Continuously, on its own schedule | One per name |
| `LifecycleEntry` | Agent startup / shutdown (`setup` / `teardown`) | One per name |

The capability registry types are the *declarative* form the loader produces from `@hook` / `@background_task` / `@capability` decorators; a `@capability` class *is* a `LifecycleEntry`.

**Three production patterns covered.**

- **Metrics** — counters and durations, exposed via `snapshot()`. Priority 200.
- **Structured logger** — fans events out to a pluggable sink (`InMemorySink` here, `JsonlSink` / `SignedChainSink` from `arctrust.audit` in production).
- **Streaming** — bounded `asyncio.Queue` + background drain task, never blocks the hot path; explicit `aclose()` to cancel the drain.

**Failure containment.** Handler crashes and handler timeouts are caught, logged, and isolated — they never raise out of `bus.emit`. A `@capability` `setup` failure is the exception: the loader emits `capability:setup_failed`, tears down set-up siblings in reverse, and **re-raises** (fail-loud). `ModuleBusError` exists for *callers* to raise when they want fail-loud semantics; the bus itself never throws it.

**API recap** — the surface this notebook exercised:

- **`arcagent.core.module_bus`** — `ModuleBus` (`subscribe`, `emit`, `handler_count`, `handler_count_by_module`), `EventContext`, `ModuleContext`.
- **`arcagent.capabilities.capability_registry`** — `HookEntry`, `BackgroundTaskEntry`, `LifecycleEntry`.
- **`arcagent.tools`** — `capability` (and `hook`, `background_task`) decorators.
- **`arcagent`** — `ToolVetoedError`, `ModuleBusError`.

**Where to next.**

- For the operational story of how arctrust audit sinks slot into a logging extension, see [`arctrust/04-audit-sinks.ipynb`](../arctrust/04-audit-sinks.ipynb).
- For the policy pipeline that runs *before* the bus on every tool call, see [`arctrust/03-policy-pipeline.ipynb`](../arctrust/03-policy-pipeline.ipynb).
- For the run loop that produces most of these events (`arcrun.run`, the `tool.start` / `tool.end` / `turn.start` / `turn.end` bridge), see [`arcrun/01-core-react.ipynb`](../arcrun/01-core-react.ipynb).